# Macaroons Tutorial

Welcome to the Macaroons tutorial. We'll build Macaroons from first principles.

## Stage 1: The "Manual" HMAC Primitive

**Goal:** Understand why `Hash(Key+Message)` is insecure and implement the actual HMAC algorithm ($K \oplus opad$ and $K \oplus ipad$).

**Task:** Code a function that takes a secret key and a message, applies the two rounds of XOR padding, and returns a SHA-256 hash.

In [ ]:
import hashlib

def manual_hmac(key: bytes, message: bytes) -> bytes:
    # TODO: Implement HMAC-SHA256
    # hash( (key ^ opad) || hash( (key ^ ipad) || message ) )
    block_size = 64
    
    # 1. Prepare the key
    if len(key) > block_size:
        key = hashlib.sha256(key).digest()
    if len(key) < block_size:
        key = key + b'\x00' * (block_size - len(key))
        
    # 2. Create padded keys
    o_key_pad = bytes((x ^ 0x5c) for x in key)
    i_key_pad = bytes((x ^ 0x36) for x in key)
    
    # 3. Inner hash
    inner_hash = hashlib.sha256(i_key_pad + message).digest()
    
    # 4. Outer hash
    return hashlib.sha256(o_key_pad + inner_hash).digest()

import hmac
# Test your implementation against Python's built-in hmac
secret = b'super_secret_key'
msg = b'Hello Macaroons'
assert manual_hmac(secret, msg) == hmac.new(secret, msg, hashlib.sha256).digest()
print("Stage 1 Complete: HMAC implementation matches built-in!")

## Stage 2: The Macaroon "Mint"

**Goal:** Create the "Root Macaroon."

**Task:** Using your manual HMAC, sign a `token_id` (e.g., "Manager-Key-001") using a `master_secret`. This becomes `Signature0`.

In [ ]:
def mint_macaroon(token_id: str, master_secret: bytes) -> bytes:
    # TODO: Create Signature0 using the token_id as the initial message
    return manual_hmac(master_secret, token_id.encode('utf-8'))

master_secret = b'this_is_the_backend_secret'
token_id = 'Manager-Key-001'
sig0 = mint_macaroon(token_id, master_secret)
print(f"Root Macaroon Signature: {sig0.hex()}")

## Stage 3: Recursive Attenuation (The Floor Manager)

**Goal:** Learn how to "shave down" the key without the original master secret.

**Task:** Pass `Signature0` to a new function that treats it as the key for a new HMAC, with the message `floor=1`. This produces `Signature1`.

In [ ]:
def add_caveat(previous_signature: bytes, caveat: str) -> bytes:
    # TODO: Create the next signature treating previous_signature as the key
    return manual_hmac(previous_signature, caveat.encode('utf-8'))

caveat1 = 'floor=1'
sig1 = add_caveat(sig0, caveat1)
print(f"Attenuated Signature (floor=1): {sig1.hex()}")

## Stage 4: The Friend's Guest Pass

**Goal:** Deepen the chain.

**Task:** Use `Signature1` as the key for a third HMAC with the message `room=breakroom`. This creates `Signature2`.

In [ ]:
caveat2 = 'room=breakroom'
sig2 = add_caveat(sig1, caveat2)
print(f"Final Attenuated Signature (room=breakroom): {sig2.hex()}")

## Stage 5: The "Door Lock" Verifier

**Goal:** Prove the integrity of the chain.

**Task:** Write a function that takes the `master_secret` and the list of caveats, re-calculates every HMAC in order, and asserts that the final result equals the Friend's `Signature2`.

In [ ]:
def verify_macaroon(master_secret: bytes, token_id: str, caveats: list, provided_signature: bytes) -> bool:
    # TODO: Re-calculate the chain from the master_secret and verify it matches provided_signature
    current_sig = mint_macaroon(token_id, master_secret)
    for caveat in caveats:
        current_sig = add_caveat(current_sig, caveat)
    
    # In cryptography, always use constant-time comparison (hmac.compare_digest) to prevent timing attacks
    return hmac.compare_digest(current_sig, provided_signature)

is_valid = verify_macaroon(master_secret, token_id, [caveat1, caveat2], sig2)
print(f"Is the valid macaroon verified? {is_valid}")
assert is_valid, "Verification should succeed!"

## Stage 6: The "Attacker" Suite (Tests)

**Goal:** Attempt to break the system.

**Task:** Write tests for:
- **Deletion:** Does the signature fail if the user removes `floor=1`?
- **Reordering:** Does it fail if the user swaps the order of caveats?
- **Forging:** Can a user add `admin=true` without knowing the previous signature?

In [ ]:
# 1. Deletion Attack: User removes 'floor=1' to try to access the whole building
is_valid_deletion = verify_macaroon(master_secret, token_id, [caveat2], sig2) # Missing caveat1
assert not is_valid_deletion, "Deletion attack should fail!"

# 2. Reordering Attack: User swaps the order, maybe caveats are processed differently in the application
is_valid_reorder = verify_macaroon(master_secret, token_id, [caveat2, caveat1], sig2)
assert not is_valid_reorder, "Reordering attack should fail!"

# 3. Forging Attack: User tries to add 'admin=true' but doesn't have sig2 to use as the key
forged_caveat = 'admin=true'
# Without sig2, they might try to guess a signature or use an old one
forged_sig = b'fake_signature'
is_valid_forge = verify_macaroon(master_secret, token_id, [caveat1, caveat2, forged_caveat], forged_sig)
assert not is_valid_forge, "Forging attack should fail!"

print("All attacker tests passed (attacks failed appropriately)!")